# Political Reddit Scraper Driven by News API

In [1]:
%pip install praw pandas requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.3/189.3 kB 4.3 MB/s eta 0:00:00


In [2]:
from datetime import datetime, timezone
from itertools import islice
import time

import pandas as pd
import praw
import requests



## Step 2: Configure APIs and Collection Targets
The notebook uses your **NewsData.io** API key to pull recent political news, turns those headlines into search queries, then searches multiple political subreddits for related Reddit discussions.

In [3]:
REDDIT_CLIENT_ID = "ANAhS6ga7S-dXxS7mxsn7w"
REDDIT_CLIENT_SECRET = "Dwowr2WXUgqGDJkHbOI2Sx1Jws7DfA"
REDDIT_USER_AGENT = "social_analytics_project/1.0"
NEWS_API_KEY = "pub_b3c903e5cc6c47f791e8190a0cd7850e"

In [5]:
POLITICAL_SUBREDDITS = [
    "politics",
    "PoliticalDiscussion",
    "Conservative",
    "Liberal",
    "democrats",
    "Republican",
    "worldnews",
    "news",
]

TARGET_POSTS = 220
COMMENTS_PER_POST = 20
NEWS_ARTICLE_LIMIT = 12
SEARCH_LIMIT_PER_QUERY = 40
REQUEST_DELAY_SECONDS = 1

In [6]:
reddit = praw.Reddit(
    client_id=REDDIT_CLIENT_ID,
    client_secret=REDDIT_CLIENT_SECRET,
    user_agent=REDDIT_USER_AGENT,
    check_for_async=False,
    ratelimit_seconds=120,
    timeout=30,
    read_only=True,
)

print(f"Read-only mode: {reddit.read_only}")
print(f"Target post count: {TARGET_POSTS}")
print(f"Political subreddits: {', '.join(POLITICAL_SUBREDDITS)}")

Read-only mode: True
Target post count: 220
Political subreddits: politics, PoliticalDiscussion, Conservative, Liberal, democrats, Republican, worldnews, news


## Step 3: Collect More Than 200 Political Posts and Their Comments
This step fetches current political headlines from the news API, builds Reddit search queries from those headlines, searches several political subreddits, and saves both post-level data and comment-level data. The collection stops once it reaches **more than 200 unique posts**.

In [7]:
def utc_string(timestamp):
    return datetime.fromtimestamp(timestamp, tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S")


def normalize_query(text, max_words=10, max_chars=80):
    cleaned = " ".join((text or "").replace("|", " ").split())
    if not cleaned:
        return ""
    shortened = " ".join(cleaned.split()[:max_words])
    return shortened[:max_chars].strip()


def fetch_news_contexts(api_key, article_limit=12):
    url = "https://newsdata.io/api/1/news"
    params = {
        "apikey": api_key,
        "category": "politics",
        "language": "en",
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    payload = response.json()
    results = payload.get("results", [])[:article_limit]

    if not results:
        raise ValueError("The news API did not return any political articles.")

    news_contexts = []
    seen_queries = set()

    for article in results:
        title = (article.get("title") or "").strip()
        description = (article.get("description") or "").strip()
        source_name = (article.get("source_id") or article.get("source_name") or "unknown").strip()
        article_url = article.get("link") or ""

        title_query = normalize_query(title)
        description_query = normalize_query(description, max_words=8, max_chars=60)

        for query in [title_query, description_query]:
            if query and query.lower() not in seen_queries:
                seen_queries.add(query.lower())
                news_contexts.append({
                    "search_query": query,
                    "news_title": title,
                    "news_description": description,
                    "news_source": source_name,
                    "news_url": article_url,
                })

    return news_contexts

In [8]:
news_contexts = fetch_news_contexts(NEWS_API_KEY, article_limit=NEWS_ARTICLE_LIMIT)
print(f"News-driven queries prepared: {len(news_contexts)}")
print(pd.DataFrame(news_contexts).head(10))

News-driven queries prepared: 18
                                        search_query  \
0  Theravance Biopharma (NASDAQ:TBPH) Rating Lowe...   
1  Theravance Biopharma (NASDAQ:TBPH – Get Free R...   
2  Uniti Group (NASDAQ:UNIT) Stock Price Expected...   
3    Uniti Group (NASDAQ:UNIT – Get Free Report) had   
4  Ensuring child safety is a collective responsi...   
5  Uttar Pradesh, Bihar, Rajasthan, Madhya Prades...   
6  The Home Depot, Inc. $HD Stake Decreased by Or...   
7  Orleans Capital Management Corp LA lowered its...   
8  Scholar Rock (NASDAQ:SRRK) Price Target Raised...   
9   Scholar Rock (NASDAQ:SRRK – Free Report) had its   

                                          news_title  \
0  Theravance Biopharma (NASDAQ:TBPH) Rating Lowe...   
1  Theravance Biopharma (NASDAQ:TBPH) Rating Lowe...   
2  Uniti Group (NASDAQ:UNIT) Stock Price Expected...   
3  Uniti Group (NASDAQ:UNIT) Stock Price Expected...   
4  Ensuring child safety is a collective responsi...   
5  Ensuring ch

In [9]:
posts_data = []
comments_data = []
seen_post_ids = set()

In [10]:
for subreddit_name in POLITICAL_SUBREDDITS:
    subreddit = reddit.subreddit(subreddit_name)
    print(f"Searching r/{subreddit_name} ...")

    for context in news_contexts:
        query = context["search_query"]

        try:
            submissions = subreddit.search(
                query,
                sort="new",
                time_filter="year",
                limit=SEARCH_LIMIT_PER_QUERY,
            )

            for post in submissions:
                if post.id in seen_post_ids:
                    continue

                seen_post_ids.add(post.id)
                post.comments.replace_more(limit=0)
                selected_comments = list(islice(post.comments.list(), COMMENTS_PER_POST))

                posts_data.append({
                    "post_id": post.id,
                    "title": post.title,
                    "selftext": post.selftext,
                    "score": post.score,
                    "num_comments": post.num_comments,
                    "upvote_ratio": getattr(post, "upvote_ratio", None),
                    "subreddit": str(post.subreddit),
                    "author": str(post.author),
                    "post_url": f"https://www.reddit.com{post.permalink}",
                    "external_url": post.url,
                    "search_query": query,
                    "matched_news_title": context["news_title"],
                    "matched_news_source": context["news_source"],
                    "matched_news_url": context["news_url"],
                    "created_utc": utc_string(post.created_utc),
                })

                for comment in selected_comments:
                    comments_data.append({
                        "comment_id": comment.id,
                        "post_id": post.id,
                        "subreddit": str(post.subreddit),
                        "post_title": post.title,
                        "comment_author": str(comment.author),
                        "comment_body": comment.body,
                        "comment_score": comment.score,
                        "comment_depth": comment.depth,
                        "comment_created_utc": utc_string(comment.created_utc),
                        "search_query": query,
                    })

                if len(posts_data) >= TARGET_POSTS:
                    break

            if len(posts_data) >= TARGET_POSTS:
                break

            time.sleep(REQUEST_DELAY_SECONDS)

        except Exception as exc:
            print(f"Skipped query '{query}' in r/{subreddit_name}: {exc}")

    if len(posts_data) >= TARGET_POSTS:
        break

Searching r/politics ...
Searching r/PoliticalDiscussion ...
Searching r/Conservative ...
Searching r/Liberal ...
Searching r/democrats ...


In [11]:
posts_df = pd.DataFrame(posts_data)
comments_df = pd.DataFrame(comments_data)

if posts_df.empty:
    raise ValueError("No Reddit posts were collected. Re-run the cell or widen the subreddit/query settings.")

posts_df = posts_df.sort_values("created_utc", ascending=False).reset_index(drop=True)
comments_df = comments_df.sort_values("comment_created_utc", ascending=False).reset_index(drop=True)

print(f"Total unique posts collected: {len(posts_df)}")
print(f"Total comments collected: {len(comments_df)}")
print(posts_df[["subreddit", "search_query", "title"]].head(10))

Total unique posts collected: 220
Total comments collected: 3493
             subreddit                                       search_query  \
0             politics  Trump says Iran has “surrendered” to neighbors...   
1  PoliticalDiscussion  Iran Says It Will Halt Attacks On Neighbours; ...   
2             politics  Iran Says It Will Halt Attacks On Neighbours; ...   
3         Conservative  Pezeshkian further said that neighbouring coun...   
4             politics  Iran Says It Will Halt Attacks On Neighbours; ...   
5             politics  Iran Says It Will Halt Attacks On Neighbours; ...   
6             politics  Iran Says It Will Halt Attacks On Neighbours; ...   
7             politics  Iran Says It Will Halt Attacks On Neighbours; ...   
8             politics  NEW DELHI, 07 March, 2026: India’s Joint Platform   
9             politics  Iran Says It Will Halt Attacks On Neighbours; ...   

                                               title  
0  Trump vows to hit ‘very hard’

## Step 4: Review and Export the Dataset


In [12]:
print(f"Posts dataset shape: {posts_df.shape}")
print(f"Comments dataset shape: {comments_df.shape}")
print(f"Reached more than 200 posts: {len(posts_df) > 200}")

display(posts_df.head(5))
display(comments_df[["post_id", "comment_author", "comment_body", "comment_score"]].head(5))

Posts dataset shape: (220, 15)
Comments dataset shape: (3493, 10)
Reached more than 200 posts: True


,post_id,title,selftext,score,num_comments,upvote_ratio,subreddit,author,post_url,external_url,search_query,matched_news_title,matched_news_source,matched_news_url,created_utc
0,1rnf5nn,Trump vows to hit ‘very hard’ after Iran’s pre...,,214,135,0.90,politics,brithus,https://www.reddit.com/r/politics/comments/1rn...,https://www.politico.com/news/2026/03/07/trump...,Trump says Iran has “surrendered” to neighbors...,Trump says Iran has “surrendered” to neighbors...,investing_us,https://www.investing.com/news/general-news/tr...,2026-03-07 17:04:33
1,1rnbgos,Will Gulf states reconsider their investment p...,"The war involving Israel, the United States, a...",1,6,0.56,PoliticalDiscussion,Only-Deal-881,https://www.reddit.com/r/PoliticalDiscussion/c...,https://www.reddit.com/r/PoliticalDiscussion/c...,Iran Says It Will Halt Attacks On Neighbours; ...,Iran Says It Will Halt Attacks On Neighbours; ...,ndtv,https://www.ndtvprofit.com/world/iran-says-it-...,2026-03-07 14:34:32
2,1rnamdd,Trump Says Iran Will Be ‘Hit Very Hard' On Sat...,,322,86,0.91,politics,No-Post4444,https://www.reddit.com/r/politics/comments/1rn...,https://www.huffpost.com/entry/trump-says-iran...,Iran Says It Will Halt Attacks On Neighbours; ...,Iran Says It Will Halt Attacks On Neighbours; ...,ndtv,https://www.ndtvprofit.com/world/iran-says-it-...,2026-03-07 13:57:17
3,1rlz0vm,"Here in San Diego, we were once a red city, bu...","Of course, it's hundreds of millions of dollar...",18,16,0.70,Conservative,Stockjock1,https://www.reddit.com/r/Conservative/comments...,https://www.reddit.com/r/Conservative/comments...,Pezeshkian further said that neighbouring coun...,Iran Says It Will Halt Attacks On Neighbours; ...,ndtv,https://www.ndtvprofit.com/world/iran-says-it-...,2026-03-06 00:34:22
4,1rlvtyy,Trump Says 'I Guess' Americans Should Worry Ab...,,33181,2301,0.96,politics,peoplemagazine,https://www.reddit.com/r/politics/comments/1rl...,https://people.com/trump-says-i-guess-american...,Iran Says It Will Halt Attacks On Neighbours; ...,Iran Says It Will Halt Attacks On Neighbours; ...,ndtv,https://www.ndtvprofit.com/world/iran-says-it-...,2026-03-05 22:26:04


,post_id,comment_author,comment_body,comment_score
0,1rnbgos,WhatAreYouSaying05,The “investments” were bullshit anyway. They k...,1
1,1rnbgos,Ornery-Ticket834,No one believes any of those phony investment ...,1
2,1rnbgos,BluesSuedeClues,"It's bold of you to suggest the gulf states ""i...",1
3,1rnamdd,Y0___0Y,Remember when he claimed the war against Iran ...,1
4,1rnf5nn,BannedPomegranate,This is just an Israel attack. Israel is doing...,1


In [13]:
posts_summary = posts_df.groupby("subreddit").size().sort_values(ascending=False).rename("post_count")
comments_summary = comments_df.groupby("subreddit").size().sort_values(ascending=False).rename("comment_count")

print("Posts by subreddit:")
display(posts_summary.to_frame())

print("Comments by subreddit:")
display(comments_summary.to_frame())

Posts by subreddit:


,post_count
subreddit,
PoliticalDiscussion,63
Conservative,62
politics,56
Liberal,25
democrats,14


Comments by subreddit:


,comment_count
subreddit,
PoliticalDiscussion,1197
politics,1046
Conservative,718
Liberal,356
democrats,176


In [14]:
posts_output_path = "reddit_political_posts.csv"
comments_output_path = "reddit_political_comments.csv"

posts_df.to_csv(posts_output_path, index=False, encoding="utf-8-sig")
comments_df.to_csv(comments_output_path, index=False, encoding="utf-8-sig")

print(f"Saved posts to: {posts_output_path}")
print(f"Saved comments to: {comments_output_path}")

Saved posts to: reddit_political_posts.csv
Saved comments to: reddit_political_comments.csv


In [15]:
posts_df[[
    "post_id",
    "subreddit",
    "search_query",
    "matched_news_title",
    "title",
    "num_comments",
    "created_utc",
]].head(15)

,post_id,subreddit,search_query,matched_news_title,title,num_comments,created_utc
0,1rnf5nn,politics,Trump says Iran has “surrendered” to neighbors...,Trump says Iran has “surrendered” to neighbors...,Trump vows to hit ‘very hard’ after Iran’s pre...,135,2026-03-07 17:04:33
1,1rnbgos,PoliticalDiscussion,Iran Says It Will Halt Attacks On Neighbours; ...,Iran Says It Will Halt Attacks On Neighbours; ...,Will Gulf states reconsider their investment p...,6,2026-03-07 14:34:32
2,1rnamdd,politics,Iran Says It Will Halt Attacks On Neighbours; ...,Iran Says It Will Halt Attacks On Neighbours; ...,Trump Says Iran Will Be ‘Hit Very Hard' On Sat...,86,2026-03-07 13:57:17
3,1rlz0vm,Conservative,Pezeshkian further said that neighbouring coun...,Iran Says It Will Halt Attacks On Neighbours; ...,"Here in San Diego, we were once a red city, bu...",16,2026-03-06 00:34:22
4,1rlvtyy,politics,Iran Says It Will Halt Attacks On Neighbours; ...,Iran Says It Will Halt Attacks On Neighbours; ...,Trump Says 'I Guess' Americans Should Worry Ab...,2301,2026-03-05 22:26:04
5,1rlq5a8,politics,Iran Says It Will Halt Attacks On Neighbours; ...,Iran Says It Will Halt Attacks On Neighbours; ...,AOC Says Trump is Willing to ‘Risk World War’ ...,25,2026-03-05 18:54:54
6,1rlph8z,politics,Iran Says It Will Halt Attacks On Neighbours; ...,Iran Says It Will Halt Attacks On Neighbours; ...,AOC says Trump is willing to ‘risk world war’ ...,344,2026-03-05 18:31:12
7,1rl742k,politics,Iran Says It Will Halt Attacks On Neighbours; ...,Iran Says It Will Halt Attacks On Neighbours; ...,"US will ‘rain missiles’, ‘death and destructio...",41,2026-03-05 03:47:32
8,1rk41y0,politics,"NEW DELHI, 07 March, 2026: India’s Joint Platform",India’s trade unions condemn US-Israeli aggres...,"Discussion Thread: March, 3rd 2026 - Midterm E...",2569,2026-03-03 22:42:34
9,1rk2y3j,politics,Iran Says It Will Halt Attacks On Neighbours; ...,Iran Says It Will Halt Attacks On Neighbours; ...,"Trump Says US Will Escort Oil Tankers, Offer I...",60,2026-03-03 21:59:38
